# 05 — Maintenance History

This notebook covers the **Maintenance History** item from the project roadmap: a simulated history of faults and repairs per machine, stored in structured form (not free text), plus a function that returns the history for a specific machine.

This becomes the **History Tool** in the target architecture — a future Diagnostic Agent will consult it alongside the RAG Tool (and, later, a Vision Tool) before answering. Combining them is a *later* roadmap step (the Orchestrator Agent); this notebook only builds and demonstrates the standalone tool.

In [1]:
import sys
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from factory_floor.machines import load_machines, load_maintenance_history, get_machine_history


## 1. The machine registry

`machines.csv` is a small, hand-generated, structured registry — 10 electric motors + 10 variable-frequency drives, each with an `equipment_type` (the same VFD / electric_motor vocabulary already used for the manuals), a family/model, a plant location, and an install date. No PDF, no embeddings — this is a plain lookup table, loaded with `csv.DictReader` just like `manual_sources.csv` already is in `factory_floor/ingestion.py`.

In [2]:
machines = load_machines()
motor_count = sum(1 for m in machines if m['equipment_type'] == 'electric_motor')
vfd_count = sum(1 for m in machines if m['equipment_type'] == 'VFD')
print(f'Total machines: {len(machines)} ({motor_count} electric motors, {vfd_count} VFDs)')
machines[:3]


Total machines: 20 (10 electric motors, 10 VFDs)


[{'machine_id': 'MOTOR-01',
  'equipment_type': 'electric_motor',
  'family': 'SIMOTICS SD',
  'model': '1LE1003-1DA43-4FA4',
  'location': 'Line 1 - Conveyor A',
  'install_date': '2018-04-13'},
 {'machine_id': 'MOTOR-02',
  'equipment_type': 'electric_motor',
  'family': 'SIMOTICS GP 1LE1',
  'model': '1LE1001-1DA23-2AA4',
  'location': 'Line 1 - Conveyor B',
  'install_date': '2020-09-30'},
 {'machine_id': 'MOTOR-03',
  'equipment_type': 'electric_motor',
  'family': 'SIMOTICS SD 1LE7',
  'model': '1LE7001-1DA23-1AA4',
  'location': 'Line 2 - Packaging',
  'install_date': '2019-07-26'}]

## 2. The maintenance history log

`maintenance_history.csv` holds 2-6 simulated events per machine (fault / repair / preventive_maintenance), generated once with a fixed random seed and committed as a static file — the same convention as `manual_sources.csv`, not something regenerated at runtime.

VFD events carry a real-looking Siemens SINAMICS fault code (e.g. F0001, F0003, F0011...), matching the fault-code system actually used in the ingested List Manual PDFs. Electric motors don't have that fault-code system in their manuals, so motor events use a free-text `description` instead, with `fault_code` left empty.

In [3]:
history = load_maintenance_history()
print(f'Total history events: {len(history)}')


Total history events: 77


## 3. Querying history for a specific machine

`get_machine_history(machine_id)` is the function the future History Tool will expose. Below, one VFD and one electric motor, to show the fault-code-vs-free-text distinction clearly.

In [4]:
vfd_id = next(m['machine_id'] for m in machines if m['equipment_type'] == 'VFD')
motor_id = next(m['machine_id'] for m in machines if m['equipment_type'] == 'electric_motor')

print(f'--- History for {vfd_id} (VFD) ---')
for event in get_machine_history(vfd_id):
    print(event)

print(f'\n--- History for {motor_id} (electric motor) ---')
for event in get_machine_history(motor_id):
    print(event)


--- History for VFD-01 (VFD) ---
{'event_id': '36', 'machine_id': 'VFD-01', 'event_date': '2020-01-27', 'event_type': 'repair', 'fault_code': 'F07860', 'description': 'External fault 1', 'action_taken': 'Checked external interlock wiring on the configured digital input', 'technician': 'P. Gomes', 'downtime_hours': '2.8'}
{'event_id': '37', 'machine_id': 'VFD-01', 'event_date': '2020-03-31', 'event_type': 'fault', 'fault_code': 'F30001', 'description': 'Power unit: Overcurrent', 'action_taken': 'Checked motor for short-circuit/ground fault and reviewed closed-loop control parameters and ramp settings', 'technician': 'A. Silva', 'downtime_hours': '1.0'}
{'event_id': '38', 'machine_id': 'VFD-01', 'event_date': '2023-05-12', 'event_type': 'fault', 'fault_code': 'F30950', 'description': 'Power unit: Internal software error', 'action_taken': 'Performed a POWER ON reset of the Control Unit', 'technician': 'J. Alves', 'downtime_hours': '1.1'}
{'event_id': '39', 'machine_id': 'VFD-01', 'event_d

## Milestone checkpoint

`machines.csv` + `maintenance_history.csv` + `get_machine_history()` are done — this is the History Tool. Wiring it into the LLM's reasoning alongside the RAG Tool (and, later, a Vision Tool) is the Orchestrator Agent's job, a separate and later roadmap step, not part of this milestone.

In the Streamlit app, the operator now picks a machine/VFD from a sidebar selector before asking a question: that selection both displays this machine's history and filters the RAG search to that machine's `equipment_type`.